In [ ]:
%%bash
pip install numpy scipy matplotlib pandas torch scikit-learn --quiet

In [ ]:
import os
import glob
import struct
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import hilbert, resample as scipy_resample
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
print('Imports OK')

In [ ]:
DATA_DIR       = '/kaggle/input/datasets/aidenerard/terracon-proceq-bridge/CJ245308_241003_001/Data/'
WORKING_DIR    = '/kaggle/working'

# Confirmed from RIS_Hi-Bright.xml
RANGE_NS       = 16.0
N_SAMPLES      = 512
NS_PER_SAMPLE  = RANGE_NS / N_SAMPLES  # 0.03125 ns/sample
EPSR           = 9.0
VELOCITY       = 0.15 / np.sqrt(EPSR)  # 0.05 m/ns in concrete
SEARCH_START   = 55    # skip surface reflection
SEARCH_END     = 150   # rebar won't be deeper than this
TARGET_SAMPLES = 512
MAX_DEPTH_CM   = 10.0  # max expected rebar cover in cm

BATCH_SIZE     = 512
EPOCHS         = 100
PATIENCE       = 15
LR             = 1e-3

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
def preprocess_trace(raw_trace, target_samples=TARGET_SAMPLES):
    '''Standardize any GPR trace to target_samples, normalized max-abs to 1.
    Format-agnostic: works for GSSI, Proceq, any system.'''
    if len(raw_trace) != target_samples:
        raw_trace = scipy_resample(raw_trace, target_samples)
    raw_trace = raw_trace - raw_trace.mean()
    max_abs = np.abs(raw_trace).max()
    if max_abs > 0:
        raw_trace = raw_trace / max_abs
    return raw_trace.astype(np.float32)


def hilbert_pick(trace, search_start=SEARCH_START, search_end=SEARCH_END):
    '''Pick horizon sample using Hilbert envelope peak in [search_start, search_end).'''
    envelope = np.abs(hilbert(trace))
    return int(np.argmax(envelope[search_start:search_end])) + search_start


def sample_to_depth_cm(sample_idx, n_samples=TARGET_SAMPLES):
    '''Convert sample index to depth in cm using XML-confirmed velocity.'''
    ns = sample_idx * (RANGE_NS / n_samples)
    depth_m = (ns * VELOCITY) / 2.0
    return depth_m * 100.0

In [ ]:
# Proceq RIS .scan binary format constants
_SCAN_MAGIC    = b'VH01SW'
_D_OFFSET      = 0x027C
_D_BLOCK_SIZE  = 0x040C   # 1036 bytes per D-block
_D_HEADER_SIZE = 16
_D_N_SAMPLES   = 510      # int16 samples per D-block payload
_D_N_REF       = 16       # first 16 D-blocks are reference sweeps (zero payload)
_D_MARKER      = b'D' + bytes(1)  # b'D\x00' D-block start tag


def read_scan_traces(scan_path):
    '''Read D-block traces from odd-numbered PRC scan file.
    Returns (N, 510) float32 array, or None if no usable data.'''
    with open(scan_path, 'rb') as f:
        raw = f.read()
    if raw[:6] != _SCAN_MAGIC:
        return None
    traces = []
    n_blocks = (len(raw) - _D_OFFSET) // _D_BLOCK_SIZE
    for i in range(n_blocks):
        block_start = _D_OFFSET + i * _D_BLOCK_SIZE
        tag = raw[block_start:block_start + 2]
        if tag != _D_MARKER:
            continue
        payload_start = block_start + _D_HEADER_SIZE
        payload = raw[payload_start:payload_start + _D_N_SAMPLES * 2]
        if len(payload) < _D_N_SAMPLES * 2:
            continue
        samples = np.frombuffer(payload, dtype=np.int16).astype(np.float32)
        traces.append(samples)
    if len(traces) <= _D_N_REF:
        return None
    return np.array(traces[_D_N_REF:])  # skip reference sweeps

In [ ]:
def load_horizon_dataset():
    '''
    Load all odd PRC scan files, pick rebar horizon using Hilbert envelope,
    return traces and normalized depth labels.

    Hilbert pseudo-labels are more reliable than TS auto-picks (which only
    cover ~10% of traces correctly).
    '''
    scan_files = sorted(glob.glob(DATA_DIR + 'PRC_*.scan'))
    odd_scans  = [f for f in scan_files
                  if int(f.split('PRC_')[1].replace('.scan', '')) % 2 == 1]

    print(f'Found {len(odd_scans)} odd scan files')

    CHANNELS_PER_SWATH = 4
    swath_groups = [odd_scans[i:i + CHANNELS_PER_SWATH]
                    for i in range(0, len(odd_scans), CHANNELS_PER_SWATH)]

    swath_traces, swath_labels, swath_ids = [], [], []

    for swath_idx, swath_scans in enumerate(swath_groups):
        traces_this, labels_this = [], []
        for scan_path in swath_scans:
            raw_traces = read_scan_traces(scan_path)
            if raw_traces is None or len(raw_traces) == 0:
                continue
            for raw in raw_traces:
                proc  = preprocess_trace(raw, TARGET_SAMPLES)
                pick  = hilbert_pick(proc)
                depth = sample_to_depth_cm(pick, TARGET_SAMPLES)
                if depth < 0.5 or depth > MAX_DEPTH_CM:
                    continue
                traces_this.append(proc)
                labels_this.append(depth / MAX_DEPTH_CM)

        if len(traces_this) == 0:
            print(f'  Swath {swath_idx + 1}: no valid traces')
            continue

        t = np.array(traces_this, dtype=np.float32)
        l = np.array(labels_this, dtype=np.float32)
        swath_traces.append(t)
        swath_labels.append(l)
        swath_ids.append(swath_idx)
        depths_in = l * MAX_DEPTH_CM / 2.54
        print(f'  Swath {swath_idx + 1}: {len(t):,} traces  '
              f'depth {depths_in.mean():.2f}in mean  '
              f'[{depths_in.min():.2f}-{depths_in.max():.2f}in]')

    print(f'\nTotal swaths loaded: {len(swath_traces)}')
    print(f'Total traces: {sum(len(t) for t in swath_traces):,}')
    return swath_traces, swath_labels, swath_ids, len(swath_traces)


swath_traces, swath_labels, swath_ids, n_swaths = load_horizon_dataset()

train_traces = np.concatenate(swath_traces[:10])
train_labels = np.concatenate(swath_labels[:10])
val_traces   = np.concatenate(swath_traces[10:])
val_labels   = np.concatenate(swath_labels[10:])

# Aliases used by downstream cells (dataset/model/train/eval)
X_train, y_train = train_traces, train_labels
X_val,   y_val   = val_traces,   val_labels

print(f'\nTrain: {len(train_traces):,} traces ({len(swath_traces[:10])} swaths)')
print(f'Val:   {len(val_traces):,} traces ({len(swath_traces[10:])} swaths)')

In [ ]:
class GPRDataset(Dataset):
    def __init__(self, traces, labels, augment=False):
        self.traces  = torch.from_numpy(traces).unsqueeze(1)  # (N, 1, 512)
        self.labels  = torch.from_numpy(labels)
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.traces[idx].clone()
        y = self.labels[idx]
        if self.augment:
            # Gaussian noise std=0.01
            if torch.rand(1) < 0.5:
                x = x + torch.randn_like(x) * 0.01
            # Amplitude scale 0.9-1.1
            if torch.rand(1) < 0.5:
                x = x * (0.9 + torch.rand(1) * 0.2)
            # Time shift +-10 samples with zero padding
            if torch.rand(1) < 0.5:
                shift = torch.randint(-10, 11, (1,)).item()
                x = torch.roll(x, shift, dims=-1)
                if shift > 0:
                    x[..., :shift] = 0.0
                elif shift < 0:
                    x[..., shift:] = 0.0
        return x, y


train_ds     = GPRDataset(X_train, y_train, augment=True)
val_ds       = GPRDataset(X_val,   y_val,   augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.score = nn.Linear(channels, 1)

    def forward(self, x):
        w = torch.softmax(self.score(x.transpose(1, 2)), dim=1)
        return (x.transpose(1, 2) * w).sum(dim=1)


class HorizonCNN(nn.Module):
    '''
    Rebar horizon depth regression.
    Input:  (batch, 1, 512) normalized trace
    Output: (batch,) normalized depth 0-1
    '''
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1,   32,  7, padding=3), nn.ReLU(),
            nn.MaxPool1d(2),                               # 512->256
            nn.Conv1d(32,  64,  5, padding=2), nn.ReLU(),
            nn.MaxPool1d(2),                               # 256->128
            nn.Conv1d(64,  128, 3, padding=1), nn.ReLU(),
            nn.MaxPool1d(2),                               # 128->64
            nn.Conv1d(128, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool1d(2),                               # 64->32
        )
        self.attn = TemporalAttention(128)
        self.head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()  # output 0-1
        )

    def forward(self, x):
        return self.head(self.attn(self.conv(x))).squeeze(-1)


model = HorizonCNN().to(DEVICE)
print(f'HorizonCNN parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

best_mae       = float('inf')
patience_count = 0
history        = []

print('  Ep   TR_loss  Val_loss  Val_MAE_cm  Val_RMSE_cm  Best_MAE          LR')
print('-' * 76)

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    tr_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * len(yb)
    tr_loss /= len(train_ds)

    # Validate
    model.eval()
    val_loss, preds_v, tgts_v = 0.0, [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb)
            val_loss += criterion(pred, yb).item() * len(yb)
            preds_v.append(pred.cpu().numpy())
            tgts_v.append(yb.cpu().numpy())
    val_loss /= len(val_ds)
    preds_v = np.concatenate(preds_v)
    tgts_v  = np.concatenate(tgts_v)

    mae_cm  = float(np.abs(preds_v - tgts_v).mean()           * MAX_DEPTH_CM)
    rmse_cm = float(np.sqrt(((preds_v - tgts_v)**2).mean())   * MAX_DEPTH_CM)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    if mae_cm < best_mae:
        best_mae, patience_count = mae_cm, 0
        torch.save(model.state_dict(), f'{WORKING_DIR}/horizon_model_best.pth')
    else:
        patience_count += 1

    history.append(dict(epoch=epoch, tr_loss=tr_loss, val_loss=val_loss,
                        mae_cm=mae_cm, rmse_cm=rmse_cm))

    if epoch % 5 == 0 or epoch == 1:
        print(f'{epoch:4d}  {tr_loss:8.5f}  {val_loss:8.5f}  {mae_cm:10.3f}  '
              f'{rmse_cm:11.3f}  {best_mae:8.3f}  {lr:10.2e}')

    if patience_count >= PATIENCE:
        print(f'Early stop at epoch {epoch}')
        break

print(f'\nBest Val MAE: {best_mae:.3f} cm  ({best_mae / 2.54:.3f} in)')

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(f'{WORKING_DIR}/horizon_model_best.pth', map_location=DEVICE))
model.eval()

preds_v, tgts_v = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        preds_v.append(model(xb.to(DEVICE)).cpu().numpy())
        tgts_v.append(yb.numpy())

preds_cm = np.concatenate(preds_v) * MAX_DEPTH_CM
tgts_cm  = np.concatenate(tgts_v)  * MAX_DEPTH_CM

mae_cm  = float(np.abs(preds_cm - tgts_cm).mean())
rmse_cm = float(np.sqrt(((preds_cm - tgts_cm)**2).mean()))
mae_in  = mae_cm  / 2.54
rmse_in = rmse_cm / 2.54

# --- Scatter plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

lo, hi = tgts_cm.min(), tgts_cm.max()
axes[0].scatter(tgts_cm, preds_cm, alpha=0.02, s=1, rasterized=True)
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=1.5)
axes[0].set_xlabel('Actual depth (cm)')
axes[0].set_ylabel('Predicted depth (cm)')
axes[0].set_title(f'Predicted vs Actual\nMAE={mae_cm:.3f} cm  RMSE={rmse_cm:.3f} cm')

# --- Residual histogram ---
residuals = preds_cm - tgts_cm
axes[1].hist(residuals, bins=100, edgecolor='k', linewidth=0.3)
axes[1].axvline(0, color='r', lw=1.5)
axes[1].set_xlabel('Residual (cm)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Residual distribution\nmean={residuals.mean():.3f}  std={residuals.std():.3f} cm')

# --- Loss curves ---
ep_h = [h['epoch'] for h in history]
axes[2].plot(ep_h, [h['tr_loss']  for h in history], label='Train')
axes[2].plot(ep_h, [h['val_loss'] for h in history], label='Val')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('SmoothL1 Loss')
axes[2].set_title('Training curves')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/rebar_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

# --- B-scan overlay: first 500 val traces ---
n_show = min(500, len(X_val))
x_show = torch.from_numpy(X_val[:n_show]).unsqueeze(1).to(DEVICE)
with torch.no_grad():
    pred_show = model(x_show).cpu().numpy() * MAX_DEPTH_CM
true_show = y_val[:n_show] * MAX_DEPTH_CM

ns_per_sample = NS_PER_SAMPLE   # RANGE_NS / TARGET_SAMPLES, from XML-confirmed config
velocity_m_ns = VELOCITY

def cm_to_sample(cm_arr):
    return (cm_arr / 100.0) / velocity_m_ns * 2.0 / ns_per_sample

fig, ax = plt.subplots(figsize=(16, 4))
ax.imshow(X_val[:n_show].T, aspect='auto', cmap='gray', vmin=-0.3, vmax=0.3, origin='upper')
x_idx = np.arange(n_show)
ax.plot(x_idx, cm_to_sample(true_show), 'r-', lw=0.8, label='Hilbert pseudo-label')
ax.plot(x_idx, cm_to_sample(pred_show), 'b-', lw=0.8, label='Model prediction')
ax.set_xlabel('Trace index')
ax.set_ylabel('Sample index')
ax.set_title('B-scan overlay - first 500 val traces\nRed=Hilbert pick, Blue=Model')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/bscan_overlay.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nFinal metrics:')
print(f'  MAE:  {mae_cm:.3f} cm  ({mae_in:.3f} in)')
print(f'  RMSE: {rmse_cm:.3f} cm  ({rmse_in:.3f} in)')

In [ ]:
import shutil
shutil.copy(f'{WORKING_DIR}/horizon_model_best.pth', f'{WORKING_DIR}/horizon_model.pth')
print(f'Saved: {WORKING_DIR}/horizon_model.pth')
print(f'MAE:  {mae_cm:.3f} cm  ({mae_in:.3f} in)')
print(f'RMSE: {rmse_cm:.3f} cm  ({rmse_in:.3f} in)')